In [ ]:
import kagglehub
import gdown
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
import numpy as np
import kagglehub
import os
import warnings
from sklearn.exceptions import ConvergenceWarning


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
paths = os.path.join(path, 'Q3_data.csv')
df =  pd.read_csv(paths)

In [ ]:
pip install catboost


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df.isna().sum() # Show number of nan in each feature

In [ ]:
# Task 1: Write your code here:
missing_df = (df.isna().mean().mul(100).round(2).to_frame(name="missing_ratio"))

missing_df["level"] = missing_df["missing_ratio"].apply(
    lambda x: "High" if x >= 55 else "Low"
)

missing_df = missing_df.sort_values(by="missing_ratio", ascending=False)
missing_df


df = df.dropna() # Drop Nan rows


In [ ]:
# Task 2: Write your code here:
df.duplicated().sum() # Show number of duplicates rows
df = df.drop_duplicates()  # Drop duplicates

In [ ]:
# Task 3: Write your code here:
# Label Encoding If matter
for col in df.select_dtypes(exclude='number').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 5: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
    counts = df[target_column].value_counts()
    print("Class counts:")
    print(counts)

    print("\nNumber of classes:", counts.shape[0])

    counts.plot(kind='bar')
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()


check_target_imbalance(df, "Target")
# no not blanaced,non default is the most freq

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(verbose=0 ,  n_estimators=320, max_depth=4)
lr_f1 = []
lr_accuracy = []

for train_index, test_index in skf.split(X, y):

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
  lr_accuracy.append(accuracy)
  lr_f1.append(f1)

print (f"F1-Score : {np.mean(lr_f1)}")
print (f"Accuracy : {np.mean(lr_accuracy)}")



In [ ]:
# Task 1: Write your code here:
feature_names = X_train.columns if "X_train" in globals() else X.columns

importance = model.feature_importances_
sorted_imp = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)

features, scores = zip(*sorted_imp)
features = features[:20]
scores = scores[:20]

plt.figure(figsize=(10, 6))
plt.barh(features, scores, color="darkblue")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Model Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# P_2 is the golden feature
print("The golden feature is" , "P_2")

In [ ]:
# Task Bonus: Write your code here:
XF= df["P_2"]



In [ ]:

lr_f11 = []
lr_accuracyy = []

for train_index, test_index in skf.split(XF, y):

  # 1. Split data
  X_train, X_test = XF.iloc[train_index], XF.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_preds = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_preds)
  f1 = f1_score(y_test, y_preds, zero_division=0)

    # Store results
  lr_accuracyy.append(accuracy)
  lr_f11.append(f1)

print (f"F1-Score : {np.mean(lr_f11)}")
print (f"Accuracy : {np.mean(lr_accuracyy)}")

In [ ]:
print("For the full model:" , np.mean(lr_f11)  )
print("For the important feature model:" , np.mean(lr_f1)  )
print("For the full model:" , np.mean(lr_accuracy)  )
print("For the important feature model:" , np.mean(lr_accuracyy)  )